# FINGUARD – SHAP Explainability

SHAP-based model interpretation, summary plots, waterfall explanations, and feature importance.



SHAP
¶

In [ ]:
!pip install shap


In [ ]:
print(X_train.columns)
print(X_test.columns)


In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# FIX: handle 3D SHAP output
shap_values_fraud = shap_values[:, :, 1]

shap.summary_plot(
    shap_values_fraud,
    X_test,
    plot_type="bar",
    max_display=10,
    show=False,
    color="#7b2d8b"
)

plt.show()


In [ ]:
import matplotlib.colors as mcolors

# Custom sky blue → purple colormap
sky_to_purple = mcolors.LinearSegmentedColormap.from_list(
    "sky_to_purple",
    ["#00bfff", "#4169e1", "#7b2d8b", "#3d0066"]
)

plt.figure(figsize=(12, 7))
shap.summary_plot(
    shap_values_fraud,
    X_test,
    max_display=10,
    show=False,
    cmap=sky_to_purple,
    alpha=0.85
)
ax = plt.gca()

# Clean white background
ax.set_facecolor("white")
plt.gcf().set_facecolor("white")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#dddddd')
ax.spines['left'].set_color('#dddddd')
ax.tick_params(colors='#444444', labelsize=11)
ax.xaxis.label.set_color('#444444')
ax.yaxis.label.set_color('#444444')
ax.set_axisbelow(True)
ax.xaxis.grid(True, color='#eeeeee', linewidth=0.8)

# Fix colorbar
colorbar = plt.gcf().axes[-1]
colorbar.tick_params(colors='#444444', labelsize=9)
colorbar.yaxis.label.set_color('#444444')
colorbar.spines['outline'].set_color('#dddddd')

plt.title("SHAP Beeswarm — Feature Impact Direction on Fraud Prediction",
          fontsize=14, fontweight='bold', color='#2d2d2d', pad=15)
plt.xlabel("SHAP Value (Impact on Fraud Prediction)", fontsize=12,
           color='#444444')
plt.tight_layout()
plt.show()


In [ ]:
#FRAUD TRANSACTION WATERFALL
fraud_indices    = np.where(y_test == 1)[0]
fraud_sample_idx = fraud_indices[0]

print("=" * 55)
print("   FRAUD TRANSACTION EXPLANATION")
print("=" * 55)
print(f"   Actual:    FRAUD")
print(f"   Predicted: {'FRAUD' if y_pred_rf[fraud_sample_idx] == 1 else 'NORMAL'}")
print("=" * 55)

plt.figure(figsize=(13, 7))
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value[1],
    shap_values_fraud[fraud_sample_idx],
    feature_names=X_test.columns.tolist(),
    max_display=10,
    show=False
)
ax = plt.gca()
ax.set_facecolor("#fff6f6")
plt.gcf().set_facecolor("#fff6f6")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#dddddd')
ax.spines['left'].set_color('#dddddd')
ax.tick_params(colors='#444444', labelsize=11)
ax.xaxis.label.set_color('#444444')

# ── Add red/blue legend ───────────────────────────────────────────
import matplotlib.patches as mpatches
red_patch  = mpatches.Patch(color='#e05c8a',  label='Pushes toward FRAUD')
blue_patch = mpatches.Patch(color='#4169e1',  label='Pushes toward NORMAL')
ax.legend(
    handles=[red_patch, blue_patch],
    loc='lower right',
    fontsize=10,
    framealpha=0.9,
    facecolor='white',
    edgecolor='#dddddd'
)

plt.title("Why This Transaction Was Flagged as FRAUD",
          fontsize=15, fontweight='bold', color='#c0392b', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
#NORMAL TRANSCATION WATERFALL
normal_indices    = np.where(y_test == 0)[0]
normal_sample_idx = normal_indices[0]

print("=" * 55)
print("   NORMAL TRANSACTION EXPLANATION")
print("=" * 55)
print(f"   Actual:    NORMAL")
print(f"   Predicted: {'FRAUD' if y_pred_rf[normal_sample_idx] == 1 else 'NORMAL'}")
print("=" * 55)

plt.figure(figsize=(13, 7))
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value[1],
    shap_values_fraud[normal_sample_idx],
    feature_names=X_test.columns.tolist(),
    max_display=10,
    show=False
)
ax = plt.gca()
ax.set_facecolor("#f6f0ff")
plt.gcf().set_facecolor("#f6f0ff")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#ccc')
ax.spines['left'].set_color('#ccc')
ax.tick_params(colors='#333')
ax.xaxis.label.set_color('#333')
ax.yaxis.label.set_color('#333')

for patch in ax.patches:
    if patch.get_facecolor()[0] > 0.5:
        patch.set_facecolor('#a44daf')
    else:
        patch.set_facecolor('#5b8dee')
    patch.set_edgecolor('white')
    patch.set_linewidth(0.5)

red_patch  = mpatches.Patch(color='#e05c8a', label='Pushes toward FRAUD')
blue_patch = mpatches.Patch(color='#4169e1', label='Pushes toward NORMAL')
ax.legend(
    handles=[red_patch, blue_patch],
    loc='lower right',
    fontsize=10,
    framealpha=0.9,
    facecolor='white',
    edgecolor='#dddddd'
)
plt.title("Why This Transaction Was Classified as NORMAL",
          fontsize=15, fontweight='bold', color='#7b2d8b', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
#FEARURE IMPORTANCE COMPARISON
import pandas as pd
import seaborn as sns

mean_shap = np.abs(shap_values_fraud).mean(axis=0)
shap_df = pd.DataFrame({
    'Feature':    X_test.columns,
    'SHAP Value': mean_shap
}).sort_values('SHAP Value', ascending=False).head(10)

palette = ["#7b2d8b","#8e3a9d","#a44daf","#c266c2","#d97fd9",
           "#e8a0e8","#f4bff4","#f9c8e8","#fad4c8","#fce8b0"]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor("#f9f6ff")
fig.suptitle("SHAP Feature Importance Analysis",
             fontsize=18, fontweight='bold', color='#3d0066', y=1.02)

# ── Left: horizontal bar ──────────────────────────────────────────
axes[0].set_facecolor("#f9f6ff")
axes[0].barh(
    shap_df['Feature'][::-1],
    shap_df['SHAP Value'][::-1],
    color=palette[::-1],
    edgecolor='white',
    linewidth=0.8,
    height=0.6
)
axes[0].set_title("Mean |SHAP Value| per Feature",
                  fontsize=13, fontweight='bold', color='#3d0066', pad=12)
axes[0].set_xlabel("Mean |SHAP Value|", fontsize=11, color='#333')
axes[0].tick_params(colors='#333')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].spines['bottom'].set_color('#ccc')
axes[0].spines['left'].set_color('#ccc')
axes[0].axvline(x=shap_df['SHAP Value'].mean(),
                color='#c266c2', linestyle='--',
                alpha=0.7, label='Average')
axes[0].legend(fontsize=10, facecolor='#f9f6ff', edgecolor='#ccc')
for i, val in enumerate(shap_df['SHAP Value'][::-1]):
    axes[0].text(val + 0.0005, i, f'{val:.4f}',
                 va='center', fontsize=9, color='#333')

# ── Right: donut chart ────────────────────────────────────────────
axes[1].set_facecolor("#f9f6ff")
wedges, texts, autotexts = axes[1].pie(
    shap_df['SHAP Value'],
    labels=shap_df['Feature'],
    autopct='%1.1f%%',
    startangle=140,
    colors=palette,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2, 'width': 0.65},
    textprops={'color': '#333', 'fontsize': 9}
)
for autotext in autotexts:
    autotext.set_color('#3d0066')
    autotext.set_fontsize(8)
    autotext.set_fontweight('bold')

centre_circle = plt.Circle((0, 0), 0.35, color='#f9f6ff')
axes[1].add_artist(centre_circle)
axes[1].text(0, 0, 'SHAP\nImpact', ha='center', va='center',
             fontsize=11, color='#3d0066', fontweight='bold')

axes[1].set_title("Relative Feature Contribution to Fraud Detection",
                  fontsize=13, fontweight='bold', color='#3d0066', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
#SHAP Analysis
import shap
import numpy as np

explainer = shap.TreeExplainer(rf_real)
shap_values = explainer.shap_values(X_test_r)

# Fix for new SHAP versions
if isinstance(shap_values, list):
    shap_values_fraud = shap_values[1]
else:
    shap_values_fraud = shap_values[:, :, 1]


In [ ]:
plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values_fraud,
    X_test_r,
    plot_type="bar",
    max_display=10
)
